# Corporate Sales Record Cleaner
Automated data sanitization pipeline using Pandas and NumPy.

This notebook loads `raw_sales.csv`, reports data quality issues, removes duplicates, imputes missing values, standardizes text, and exports cleaned data plus summary statistics.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

input_path = Path('raw_sales.csv')
df = pd.read_csv(input_path)
print('Original shape:', df.shape)
display(df.head())

In [ ]:
# Baseline audit
print('Missing values before cleaning:')
display(df.isnull().sum().to_frame('missing_count'))
print('Exact duplicate rows:', df.duplicated().sum())
df.describe(include='all').transpose().to_csv('baseline_summary_statistics.csv')

In [ ]:
# Remove exact duplicate rows
cleaned = df.drop_duplicates().copy()

# Standardize text values
text_columns = cleaned.select_dtypes(include=['object', 'string']).columns
for column in text_columns:
    cleaned[column] = cleaned[column].astype('string').str.strip().str.lower()
    cleaned[column] = cleaned[column].replace({'': pd.NA, 'nan': pd.NA})

# Restore readable casing for category columns
if 'City' in cleaned.columns:
    cleaned['City'] = cleaned['City'].str.title()
if 'Payment_Method' in cleaned.columns:
    cleaned['Payment_Method'] = cleaned['Payment_Method'].str.title()

# Standardize date format
if 'Order_Date' in cleaned.columns:
    cleaned['Order_Date'] = pd.to_datetime(cleaned['Order_Date'], errors='coerce').dt.strftime('%Y-%m-%d')

# Impute missing values: median for numeric, mode for text/category
for column in cleaned.columns:
    if cleaned[column].isna().any():
        if pd.api.types.is_numeric_dtype(cleaned[column]):
            cleaned[column] = cleaned[column].fillna(cleaned[column].median())
        else:
            modes = cleaned[column].mode(dropna=True)
            cleaned[column] = cleaned[column].fillna(modes.iloc[0] if not modes.empty else 'Unknown')

for column in ['Price', 'Quantity', 'Age', 'Order_ID']:
    if column in cleaned.columns:
        cleaned[column] = pd.to_numeric(cleaned[column], errors='coerce')
for column in ['Price', 'Quantity', 'Age']:
    if column in cleaned.columns:
        if cleaned[column].isna().any():
            cleaned[column] = cleaned[column].fillna(cleaned[column].median())
        cleaned[column] = cleaned[column].clip(lower=0)

cleaned.to_csv('cleaned_sales.csv', index=False)
cleaned.describe(include='all').transpose().to_csv('cleaned_summary_statistics.csv')
print('Cleaned shape:', cleaned.shape)
print('Missing values after cleaning:')
display(cleaned.isnull().sum().to_frame('missing_count'))
display(cleaned.head())

## Interpretation
- Exact duplicate rows are removed.
- Numeric missing values are filled with the median.
- Categorical/text missing values are filled with the mode (or `Unknown` if no mode exists).
- Text is trimmed and category casing is standardized.
- The cleaned CSV and before/after summary statistics are saved in the project folder.